# 01 The random split

A dead end, kept. The first payments model was graded on a random split of transactions, the way most fraud notebooks start. It looked excellent. Graded on a split by time, the way the bank would have met it, it did not.

The reason is the shape of card fraud: a compromised card produces a run of fraud over days. A random split puts some of the run in training and the rest in test, so the model is graded on transactions it has, in effect, already seen. This notebook runs both splits on the fixture slice (data/fixtures, built by scripts/build_fixture_slice.py) so it runs in CI in seconds; the full desk is graded only by time (evaluation/plan.yaml).

In [1]:
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from pathlib import Path

LAKE = Path("../data/lake/fixture/spark-warehouse")
def read(name):
    return pq.read_table(LAKE / name).to_pandas()

In [2]:
from friction_features import payments as pf
feats = read("payments_features")
silver = read("payments_silver")[["txn_ref", "account_key", "event_ts"]]
hist = read("labels_payments_history")
latest = hist[hist["__END_AT"].isna()][["event_ref", "is_fraud"]].rename(columns={"event_ref": "txn_ref"})
df = silver.merge(feats, on="txn_ref").merge(latest, on="txn_ref", how="left")
df["y"] = df["is_fraud"].fillna(False).astype(int)
df = df.sort_values(["event_ts", "txn_ref"]).reset_index(drop=True)
X = df[list(pf.FEATURES)].to_numpy(float)
print(len(df), "transactions,", int(df.y.sum()), "frauds,", df.account_key.nunique(), "cards")

/tmp/ipykernel_31958/3726946769.py:7: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["y"] = df["is_fraud"].fillna(False).astype(int)


32353 transactions, 90 frauds, 8 cards


In [3]:
from friction_models.learners import Gbm
from friction_evaluation.grade import roc_auc

def fit_and_score(train, test):
    g = Gbm.fit(X[train], df.y.to_numpy()[train], list(pf.FEATURES))
    return roc_auc(g.predict(X[test]), df.y.to_numpy()[test].astype(bool))

rng = np.random.default_rng(20261029)
random_test = rng.random(len(df)) < 0.3
auc_random = fit_and_score(~random_test, random_test)
cut = df.event_ts.quantile(0.7)
time_test = (df.event_ts >= cut).to_numpy()
auc_time = fit_and_score(~time_test, time_test)
print(f"random split ROC AUC {auc_random:.3f}; time split ROC AUC {auc_time:.3f}")

random split ROC AUC 0.968; time split ROC AUC 0.806


What to take from it: the gap between the two numbers is the leak. The registered plan splits by time with gaps between train, validation and test, reads labels as known at each cut, and resamples whole cards in the bootstrap, so a run of fraud on one card never sits on both sides of a line.